# Medical Insurance Cost Prediction — Explainable Machine Learning

**Objective:** Estimate annual medical charges for new customers using demographic and lifestyle information.

**Primary requirement:** Predictions should be understandable enough to support transparent decision-making.

### Project workflow

1. Load and understand the data
2. Check data quality
3. Explore relationships with medical charges
4. Engineer features based on observed relationships
5. Split data into training, validation, and test sets
6. Select features using the validation set
7. Train and compare Linear Regression and Random Forest
8. Tune Random Forest as a benchmark
9. Evaluate the final chosen model on the held-out test set
10. Analyze errors and model limitations
11. Retrain a deployment version on all available data
12. Create a reusable prediction function

## 1. Problem Definition

ACME Insurance wants to estimate a customer's annual medical expenditure from information available at signup.

The available variables are:

- `age`
- `sex`
- `bmi`
- `children`
- `smoker`
- `region`

The target variable is:

- `charges` — annual medical charges

This is a **regression problem** because the target is a continuous numerical value.

> **Important scope note:** This is a proof-of-concept using a small public dataset. It should not be treated as a production insurance-pricing system without additional actuarial, regulatory, fairness, privacy, and validation work.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)

DATA_URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
display(df.head())

## 2. Data Quality Check

The original dataset contains 1,338 rows. One duplicate row was identified and removed, leaving **1,337 unique records**.

No missing values were found in the available columns.

In [ ]:
print("Missing values:")
display(df.isna().sum())

print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Shape after duplicate removal:", df.shape)

In [ ]:
display(df.describe())

### Initial observations

After duplicate removal:

- Records: **1,337**
- Features available before engineering: **6**
- Target: `charges`
- `charges` is strongly right-skewed (skewness ≈ **1.515**).
- The mean charge is about **$13,279**, while the median is about **$9,386**, showing the influence of high-cost customers.

Skewness is useful for understanding the target distribution, but it does **not** by itself mean that a log-transformed target will produce a better predictive model. Model performance must be judged using the actual prediction metrics.

In [ ]:
print("Charges skewness:", df["charges"].skew())
print("Mean charges:", df["charges"].mean())
print("Median charges:", df["charges"].median())

## 3. Exploratory Data Analysis

The main purpose of EDA is to identify relationships that may be useful for prediction.

The analysis found that smoking status has a very strong relationship with charges. BMI and age also show meaningful relationships, particularly among smokers.

Region shows smaller differences, while sex provides little additional predictive value after considering the other variables.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["charges"], bins=30)
ax.set_title("Distribution of Annual Medical Charges")
ax.set_xlabel("Charges")
ax.set_ylabel("Number of Customers")
plt.show()

In [ ]:
smoker_means = df.groupby("smoker")["charges"].mean().sort_values(ascending=False)
display(smoker_means)

fig, ax = plt.subplots(figsize=(7, 5))
smoker_means.plot(kind="bar", ax=ax)
ax.set_title("Average Charges by Smoking Status")
ax.set_xlabel("Smoking Status")
ax.set_ylabel("Average Charges")
plt.xticks(rotation=0)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for smoker_value, group in df.groupby("smoker"):
    ax.scatter(group["bmi"], group["charges"], label=smoker_value, alpha=0.6)
ax.set_title("BMI vs Medical Charges")
ax.set_xlabel("BMI")
ax.set_ylabel("Charges")
ax.legend(title="Smoker")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for smoker_value, group in df.groupby("smoker"):
    ax.scatter(group["age"], group["charges"], label=smoker_value, alpha=0.6)
ax.set_title("Age vs Medical Charges")
ax.set_xlabel("Age")
ax.set_ylabel("Charges")
ax.legend(title="Smoker")
plt.show()

In [ ]:
region_means = df.groupby("region")["charges"].mean().sort_values(ascending=False)
display(region_means)

## 4. Feature Engineering

The EDA suggested that the relationship between smoking and BMI is not simply additive.

A smoker's BMI is therefore represented with an interaction feature:

\[
smoker\_bmi = smoker\_flag 	imes BMI
\]

where:

- `smoker_flag = 1` for a smoker
- `smoker_flag = 0` for a non-smoker

This lets Linear Regression model a different BMI contribution for smokers.

A `smoker_age` interaction was also tested but was removed because it did not improve validation performance.

`sex` was also removed because removing it slightly improved validation performance and it provided little additional predictive value given the remaining features.

Region is one-hot encoded with the northeast region as the reference category.

In [ ]:
# Encode binary variables.
df["smoker_flag"] = df["smoker"].map({"yes": 1, "no": 0})

# Interaction feature supported by EDA.
df["smoker_bmi"] = df["smoker_flag"] * df["bmi"]

# One-hot encode region; northeast becomes the reference category.
df = pd.get_dummies(
    df,
    columns=["region"],
    drop_first=True,
    dtype=int
)

display(df.head())

### Final feature set

The final Linear Regression model uses:

- `age`
- `bmi`
- `children`
- `smoker_flag`
- `smoker_bmi`
- `region_northwest`
- `region_southeast`
- `region_southwest`

Excluded:

- `charges` — target
- `smoker` — replaced by `smoker_flag`
- `smoker_age` — did not improve validation performance
- `sex` — did not provide useful additional predictive value

No scaling is required for ordinary Linear Regression here. The raw numerical units also make the final coefficients easier to interpret in dollar terms.

In [ ]:
feature_columns = [
    "age",
    "bmi",
    "children",
    "smoker_flag",
    "smoker_bmi",
    "region_northwest",
    "region_southeast",
    "region_southwest",
]

X = df[feature_columns].copy()
y = df["charges"].copy()

print("Features used:")
print(feature_columns)
print("\nX shape:", X.shape)
print("y shape:", y.shape)

## 5. Train / Validation / Test Split

The data is split into:

- **70% training** — used to fit models
- **15% validation** — used for feature/model decisions
- **15% test** — held back for final evaluation

The test set should not be used to repeatedly make modeling decisions.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

## 6. Feature Selection Experiments

Feature selection was driven by EDA rather than trying every possible combination blindly.

The important findings were:

- Removing `smoker_bmi` caused a large deterioration in validation performance, so it was retained.
- Removing `smoker_age` did not improve the model, so it was excluded.
- Removing `sex` slightly improved validation performance, so it was excluded.

In [ ]:
def evaluate_regression(model, X_fit, y_fit, X_eval, y_eval):
    model.fit(X_fit, y_fit)
    predictions = model.predict(X_eval)
    return {
        "R2": r2_score(y_eval, predictions),
        "MAE": mean_absolute_error(y_eval, predictions),
        "RMSE": root_mean_squared_error(y_eval, predictions),
    }

# Compare the final feature set against removing the interaction.
X_no_smoker_bmi = X.drop(columns=["smoker_bmi"])

X_nb_train = X_no_smoker_bmi.loc[X_train.index]
X_nb_val = X_no_smoker_bmi.loc[X_val.index]

interaction_model = LinearRegression()
interaction_model.fit(X_train, y_train)
interaction_preds = interaction_model.predict(X_val)

no_interaction_model = LinearRegression()
no_interaction_model.fit(X_nb_train, y_train)
no_interaction_preds = no_interaction_model.predict(X_nb_val)

comparison = pd.DataFrame([
    {
        "Feature set": "With smoker_bmi",
        "R2": r2_score(y_val, interaction_preds),
        "MAE": mean_absolute_error(y_val, interaction_preds),
        "RMSE": root_mean_squared_error(y_val, interaction_preds),
    },
    {
        "Feature set": "Without smoker_bmi",
        "R2": r2_score(y_val, no_interaction_preds),
        "MAE": mean_absolute_error(y_val, no_interaction_preds),
        "RMSE": root_mean_squared_error(y_val, no_interaction_preds),
    }
])

display(comparison)

## 7. Baseline: Linear Regression

Linear Regression is an appropriate baseline because it is:

- simple,
- fast,
- easy to interpret,
- suitable for a continuous target,
- naturally compatible with coefficient-based explanations.

The engineered `smoker_bmi` term allows the model to represent an interaction that plain additive Linear Regression would otherwise miss.

In [ ]:
linear_model = LinearRegression()

linear_results = evaluate_regression(
    linear_model,
    X_train, y_train,
    X_val, y_val
)

display(pd.DataFrame([linear_results], index=["Linear Regression"]))

## 8. Cross-Validation for the Linear Model

Cross-validation checks whether model performance is reasonably stable across multiple train/validation folds.

Here it is performed on the **training portion only**, so the held-out test set remains untouched.

Because scikit-learn returns negative MAE when using `neg_mean_absolute_error`, the scores are converted back to positive MAE values for interpretation.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_val_score(
    LinearRegression(),
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

cv_mae = -cv_scores

print("Fold MAEs:", np.round(cv_mae, 2))
print("Mean CV MAE:", round(cv_mae.mean(), 2))
print("Std CV MAE:", round(cv_mae.std(), 2))

## 9. Random Forest Benchmark

Random Forest was evaluated as a non-linear benchmark.

It can capture complex relationships without requiring us to specify the exact mathematical form of every relationship.

The Random Forest achieved lower validation error than Linear Regression, so it was also tuned before the final model decision.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=RANDOM_STATE
)

rf_results = evaluate_regression(
    rf,
    X_train, y_train,
    X_val, y_val
)

display(pd.DataFrame([rf_results], index=["Random Forest"]))

## 10. Random Forest Hyperparameter Tuning

`RandomizedSearchCV` evaluates randomly selected combinations of hyperparameters using 5-fold cross-validation on the training data.

The search used MAE as the optimization metric because an error stated in dollars is directly meaningful for this problem.

In [ ]:
rf_base = RandomForestRegressor(random_state=RANDOM_STATE)

param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.5, 0.7, 1.0],
}

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_distributions,
    n_iter=30,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

print("Best parameters:")
print(rf_search.best_params_)

In [ ]:
best_rf = rf_search.best_estimator_

rf_val_preds = best_rf.predict(X_val)

rf_val_results = {
    "R2": r2_score(y_val, rf_val_preds),
    "MAE": mean_absolute_error(y_val, rf_val_preds),
    "RMSE": root_mean_squared_error(y_val, rf_val_preds),
}

display(pd.DataFrame([rf_val_results], index=["Tuned Random Forest"]))

## 11. Final Model Decision

The tuned Random Forest produced lower prediction error than Linear Regression.

However, Linear Regression was selected as the final model because the project places a strong emphasis on direct, coefficient-based interpretability.

This is a deliberate trade-off:

- **Random Forest:** better predictive performance on this dataset.
- **Linear Regression:** simpler and directly interpretable in terms of feature coefficients.

The test-set comparison below is reported as an evaluation result, not as an invitation to keep changing the model.

In [ ]:
# Evaluate the selected Linear Regression model on the held-out test set.
linear_test_model = LinearRegression()
linear_test_model.fit(X_train, y_train)

linear_test_preds = linear_test_model.predict(X_test)

linear_test_results = {
    "Model": "Linear Regression",
    "R2": r2_score(y_test, linear_test_preds),
    "MAE": mean_absolute_error(y_test, linear_test_preds),
    "RMSE": root_mean_squared_error(y_test, linear_test_preds),
}

# Evaluate the tuned Random Forest on the same held-out test set.
rf_test_preds = best_rf.predict(X_test)

rf_test_results = {
    "Model": "Random Forest",
    "R2": r2_score(y_test, rf_test_preds),
    "MAE": mean_absolute_error(y_test, rf_test_preds),
    "RMSE": root_mean_squared_error(y_test, rf_test_preds),
}

test_comparison = pd.DataFrame([linear_test_results, rf_test_results])
display(test_comparison)

### Recorded test-set results from the original project run

The original notebook produced:

| Model | R² | MAE | RMSE |
|---|---:|---:|---:|
| Linear Regression | 0.8469 | $2,834 | $5,111 |
| Random Forest | 0.8743 | $2,491 | $4,631 |

Random Forest's test MAE was about **12.1% lower** than Linear Regression's.

The Linear Regression model was retained because the project prioritizes direct interpretability.

## 12. Error Analysis

The most useful question after evaluation is not only "How accurate is the model?" but also:

> **Where does the model make large mistakes?**

We inspect the largest absolute errors and look for common characteristics.

In [ ]:
error_analysis = X_test.copy()

error_analysis["actual_charge"] = y_test
error_analysis["predicted_charge"] = linear_test_preds
error_analysis["error"] = error_analysis["actual_charge"] - error_analysis["predicted_charge"]
error_analysis["absolute_error"] = error_analysis["error"].abs()

worst_cases = (
    error_analysis
    .sort_values("absolute_error", ascending=False)
    .head(10)
)

display(worst_cases)

### Important limitation

The original analysis identified a small group of non-smokers with very large prediction errors.

The analysis found **13 such validation cases** with errors above $10,000. This corresponds to approximately 6.5% of the 201 validation records examined — **not 6.5% of all non-smokers**.

The available features do not contain health history or other information that could explain every high-cost case. Therefore, possible explanations such as health conditions or prior medical history should be treated as hypotheses, not confirmed causes.

In [ ]:
# Reproduce the >$10,000 validation-error check.
val_error_table = X_val.copy()
val_error_table["actual_charge"] = y_val
val_error_table["predicted_charge"] = interaction_model.predict(X_val)
val_error_table["absolute_error"] = (
    val_error_table["actual_charge"] - val_error_table["predicted_charge"]
).abs()

high_error_cases = val_error_table[val_error_table["absolute_error"] > 10000]

print("Validation records:", len(y_val))
print("Validation errors > $10,000:", len(high_error_cases))
print("Share of validation records:", round(len(high_error_cases) / len(y_val) * 100, 2), "%")

display(high_error_cases.sort_values("absolute_error", ascending=False).head(10))

## 13. Actual vs Predicted Charges

A useful visual diagnostic is to compare actual charges with predicted charges.

Points closer to the diagonal relationship indicate smaller prediction errors.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_test, linear_test_preds, alpha=0.65)

min_value = min(y_test.min(), linear_test_preds.min())
max_value = max(y_test.max(), linear_test_preds.max())

ax.plot([min_value, max_value], [min_value, max_value], linestyle="--")

ax.set_title("Linear Regression: Actual vs Predicted Charges")
ax.set_xlabel("Actual Charges")
ax.set_ylabel("Predicted Charges")

plt.show()

## 14. Final Linear Regression Coefficients

For interpretation, the final deployment model is trained on all available cleaned data **after model selection and test evaluation are complete**.

The deployment model is not used to generate a new test score.

Coefficient interpretation should always be made while holding the other model inputs constant.

In [ ]:
deployment_model = LinearRegression()
deployment_model.fit(X, y)

coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": deployment_model.coef_
}).sort_values("coefficient", ascending=False)

display(coef_table)

print("Intercept:", deployment_model.intercept_)

### How to interpret the interaction

Because the model contains:

\[
smoker\_bmi = smoker\_flag 	imes BMI
\]

the smoking effect is **not represented by the `smoker_flag` coefficient alone**.

For a smoker, the combined smoking-related contribution is:

\[
smoker\_flag\ coefficient + smoker\_bmi\ coefficient 	imes BMI
\]

For a non-smoker, `smoker_flag = 0` and `smoker_bmi = 0`.

Therefore, it would be incorrect to describe the `smoker_flag` coefficient alone as "the total effect of smoking."

## 15. Reusable Prediction Function

The function below performs the same feature preparation used during training.

Notice that the region columns are created explicitly rather than calling `get_dummies()` on a single live row. This guarantees that a new customer receives the same feature structure the model expects.

In [ ]:
import joblib

# Save the deployment model.
joblib.dump(deployment_model, "insurance_charge_model.pkl")

def predict_charge(age, bmi, children, smoker, region):
    # Validate categorical inputs.
    if smoker not in {"yes", "no"}:
        raise ValueError("smoker must be 'yes' or 'no'")

    if region not in {"northeast", "northwest", "southeast", "southwest"}:
        raise ValueError(
            "region must be one of: northeast, northwest, southeast, southwest"
        )

    smoker_flag = 1 if smoker == "yes" else 0
    smoker_bmi = smoker_flag * bmi

    input_data = pd.DataFrame([{
        "age": age,
        "bmi": bmi,
        "children": children,
        "smoker_flag": smoker_flag,
        "smoker_bmi": smoker_bmi,
        "region_northwest": int(region == "northwest"),
        "region_southeast": int(region == "southeast"),
        "region_southwest": int(region == "southwest"),
    }])

    prediction = deployment_model.predict(input_data)[0]

    return round(float(prediction), 2)

In [ ]:
# Example prediction
prediction = predict_charge(
    age=45,
    bmi=32.1,
    children=2,
    smoker="yes",
    region="southeast"
)

print(f"Predicted annual medical charge: ${prediction:,.2f}")

## 16. Final Conclusions

### What worked

- Smoking status was the strongest observed predictor of charges.
- The `smoker_bmi` interaction materially improved Linear Regression.
- Age and number of children had positive, relatively modest coefficients.
- Region contributed smaller adjustments than smoking-related variables.
- Sex was removed because it did not provide useful additional predictive value in the validation comparison.

### Model decision

Random Forest produced lower test error, but Linear Regression was selected because it provides direct coefficient-based explanations.

### Limitations

- The dataset contains only 1,337 unique records.
- The available variables do not capture every factor affecting medical charges.
- Some non-smoking customers have very large unexplained errors.
- The findings are predictive associations, not causal estimates.
- This proof-of-concept should not be treated as a production insurance-pricing decision system without additional validation, fairness, regulatory, and actuarial review.

## 17. Project Summary

**Final selected model:** Linear Regression

**Held-out test performance:**

- R²: **0.8469**
- MAE: **$2,834**
- RMSE: **$5,111**

**Benchmark:** Tuned Random Forest achieved lower test MAE of approximately **$2,491**, about 12.1% lower than Linear Regression.

**Reason for selecting Linear Regression:** transparent coefficient-based explanations.

**Key engineered feature:** `smoker_bmi`

**Main limitation:** a subset of high-cost customers cannot be accurately explained using the available signup features.